In [ ]:
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 60.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 12.4 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659447 sha256=36c584bdf1e957e0a1c2002e88ab463a00c6e5719f86ee419882f1358b3d240e
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


In [ ]:
import numpy as np
import pycuda.driver as cuda_drv
import pycuda.autoinit
from pycuda.compiler import SourceModule
import time
import sys

In [ ]:
# CUDA-ядро: глобальная сумма вектора с блочной редукцией через shared memory

cuda_kernel_code = """
__global__ void reduce_sum_kernel(int* global_output, const int* global_input, unsigned int data_length) {
    extern __shared__ int shared_data[];

    int thread_id = threadIdx.x;
    int global_idx = blockIdx.x * blockDim.x + thread_id;

    // Загрузка элемента в shared память
    int temp = 0;
    if (global_idx < data_length) {
        temp = global_input[global_idx];
    }
    shared_data[thread_id] = temp;
    __syncthreads();

    // Редукция внутри блока (логарифмическое суммирование)
    for (int offset = blockDim.x / 2; offset > 0; offset >>= 1) {
        if (thread_id < offset) {
            shared_data[thread_id] += shared_data[thread_id + offset];
        }
        __syncthreads();
    }

    // Запись результата блока в глобальную память с атомарным сложением
    if (thread_id == 0) {
        atomicAdd(global_output, shared_data[0]);
    }
}
"""

In [ ]:
def gpu_vector_sum(data_array):
    """
    Вычисление суммы элементов вектора на GPU с помощью CUDA.
    Возвращает (сумма, время выполнения в секундах).
    """
    start = time.perf_counter()

    # Подготовка памяти на устройстве
    data_size = data_array.size
    data_bytes = data_array.nbytes
    result_bytes = np.int32().nbytes

    d_input = cuda_drv.mem_alloc(data_bytes)
    d_output = cuda_drv.mem_alloc(result_bytes)

    # Инициализация нулём на GPU
    zero = np.array([0], dtype=np.int32)
    cuda_drv.memcpy_htod(d_output, zero)

    # Копирование данных
    cuda_drv.memcpy_htod(d_input, data_array)

    # Компиляция и получение функции ядра
    module = SourceModule(cuda_kernel_code)
    kernel_func = module.get_function("reduce_sum_kernel")

    # Параметры запуска
    threads_per_block = 256
    blocks_count = (data_size + threads_per_block - 1) // threads_per_block
    shared_mem_size = threads_per_block * np.int32().itemsize

    # Запуск ядра
    kernel_func(
        d_output, d_input, np.int32(data_size),
        block=(threads_per_block, 1, 1),
        grid=(blocks_count, 1),
        shared=shared_mem_size
    )

    # Чтение результата
    result = np.empty(1, dtype=np.int32)
    cuda_drv.memcpy_dtoh(result, d_output)

    # Освобождение памяти
    d_input.free()
    d_output.free()

    end = time.perf_counter()
    return result[0], (end - start)

In [ ]:
def cpu_vector_sum(data_array):
    """
    Последовательное суммирование элементов вектора на CPU.
    Возвращает сумму.
    """
    total = 0
    for val in data_array:
        total += val
    return total

In [ ]:
def run_experiments():
    """
    Проведение замеров для различных размеров вектора и вывод таблицы.
    """
    sizes = [1000, 5000, 10000, 50000, 100000, 500000, 1000000]

    print("Размер вектора | Время CPU (с) | Время GPU (с) | Ускорение (CPU/GPU)")
    print("-" * 65)

    for n in sizes:
        # Генерация случайных целых чисел
        vector = np.random.randint(1, 101, size=n, dtype=np.int32)

        # CPU замер
        cpu_start = time.perf_counter()
        cpu_res = cpu_vector_sum(vector)
        cpu_time = time.perf_counter() - cpu_start

        # GPU замер
        gpu_res, gpu_time = gpu_vector_sum(vector)

        # Проверка корректности
        assert cpu_res == gpu_res, f"Ошибка: сумма не совпадает для n={n}"

        speedup = cpu_time / gpu_time if gpu_time > 0 else float('inf')
        print(f"{n:14d} | {cpu_time:13.6f} | {gpu_time:13.6f} | {speedup:10.2f}")


if __name__ == "__main__":

    # Запуск полных экспериментов
    run_experiments()

Размер вектора | Время CPU (с) | Время GPU (с) | Ускорение (CPU/GPU)
-----------------------------------------------------------------
          1000 |      0.000131 |      0.001267 |       0.10
          5000 |      0.000637 |      0.001976 |       0.32
         10000 |      0.001621 |      0.000604 |       2.69
         50000 |      0.004698 |      0.000593 |       7.92
        100000 |      0.008605 |      0.000614 |      14.01
        500000 |      0.045619 |      0.001000 |      45.60
       1000000 |      0.084949 |      0.001651 |      51.46
